# Notebook 08 — Cross-Domain Generalization on TrashCan 1.0 (Metal + Bio)

**Paper artifact:** TrashCan generalization table (`tab:trashcan_results`).

This notebook stress-tests the paper's detection pipeline **outside** the SINTEF LIACI corrosion domain. The Medium backbone is retrained from scratch on the public **TrashCan 1.0** dataset (Material Version, > 7k images) under the same Direct Transfer protocol used for the underwater corrosion model, and is then evaluated **with and without Test-Time Augmentation (TTA)** on the held-out validation split. The COCO annotations are filtered to two underwater-debris classes — **Metal** (corroded metallic litter) and **Bio** (marine fauna/flora) — where the *Metal* class acts as a transferable proxy for submerged corrosion.

**Key result.** Manual TTA (multi-scale + horizontal flip fused with Weighted Boxes Fusion) trades a small drop in precision-oriented metrics for a substantial **+6.28% Recall** gain on the validation set, raising F1-Score from 0.6771 to 0.6882. This confirms that the augmentation strategy generalizes across underwater domains and is not over-fitted to the corrosion dataset.

---

| | |
|---|---|
| **Inputs** | TrashCan 1.0 Material Version (COCO JSON: `instances_train_trashcan.json`, `instances_val_trashcan.json`); `yolo26m.pt` pretrained weights |
| **Output** | Standard-vs-TTA comparison table → `trashcan_tta_results.csv`; paper `tab:trashcan_results` |
| **Environment** | Ultralytics 8.4.5 · PyTorch 2.8.0+cu128 · Python 3.9.13 · NVIDIA RTX 4090 |

> **Note on outputs.** All code cells and their outputs are preserved exactly as executed on the original GPU run; the Ultralytics console logs (download, COCO→YOLO conversion, the 50-epoch training trace and validation summaries) appear in their original form. Re-running under `requirements.txt` reproduces the same table.


## 1 · Setup and dependencies

Install and import the detection stack (Ultralytics + PyTorch), the COCO tooling (`pycocotools`), and the `ensemble-boxes` library used for the Weighted Boxes Fusion step inside TTA. The helper resets the CUDA cache and reports the active GPU so every run starts from a clean memory baseline.


In [1]:
!pip install -q ultralytics gdown pycocotools opencv-python matplotlib pandas requests tqdm ensemble-boxes

In [2]:
from ensemble_boxes import weighted_boxes_fusion
import cv2
import numpy as np

def predict_with_tta(model, img_path, conf=0.25, scales=[0.9, 1.0, 1.1], use_flip=True):
    img = cv2.imread(str(img_path))
    if img is None: return np.array([]), np.array([]), np.array([])
    orig_h, orig_w = img.shape[:2]
    all_boxes = []
    all_scores = []
    all_labels = []
    weights_tta = []
    
    variants = []
    for scale in scales:
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        scaled = cv2.resize(img, (new_w, new_h))
        variants.append(('original', scale, scaled))
        if use_flip:
            flipped = cv2.flip(scaled, 1)
            variants.append(('flipped', scale, flipped))
    
    for var_type, scale, var_img in variants:
        results = model.predict(var_img, conf=conf, verbose=False)
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxyn.cpu().numpy()
            scores = results[0].boxes.conf.cpu().numpy()
            labels = results[0].boxes.cls.cpu().numpy()
            if var_type == 'flipped':
                boxes_corrected = boxes.copy()
                boxes_corrected[:, 0] = 1.0 - boxes[:, 2]
                boxes_corrected[:, 2] = 1.0 - boxes[:, 0]
                boxes = boxes_corrected
            all_boxes.append(boxes)
            all_scores.append(scores)
            all_labels.append(labels)
            weights_tta.append(1.0 if scale == 1.0 else 0.8)
        else:
            all_boxes.append(np.array([]).reshape(0, 4))
            all_scores.append(np.array([]))
            all_labels.append(np.array([]))
            weights_tta.append(0.8)
            
    if len(all_boxes) > 0 and any(len(b) > 0 for b in all_boxes):
        all_boxes = [b if len(b) > 0 else np.array([]).reshape(0, 4) for b in all_boxes]
        try:
            boxes_fused, scores_fused, labels_fused = weighted_boxes_fusion(
                all_boxes, all_scores, all_labels, weights=weights_tta, iou_thr=0.5, skip_box_thr=0.001
            )
            return boxes_fused, scores_fused, labels_fused
        except:
            pass
    return np.array([]), np.array([]), np.array([])

class MAPCalculator:
    def __init__(self):
        self.predictions = []
        self.targets = []
    def update(self, pred_boxes, pred_scores, target_boxes):
        self.predictions.append({'boxes': pred_boxes, 'scores': pred_scores})
        self.targets.append(target_boxes)
    def compute_iou(self, box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        union_area = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter_area
        return inter_area / union_area if union_area > 0 else 0
    def compute_ap(self, iou_threshold):
        all_scores, all_tps, total_gt = [], [], 0
        for preds, targets in zip(self.predictions, self.targets):
            total_gt += len(targets)
            if len(preds['boxes']) == 0: continue
            p_boxes, p_scores = preds['boxes'], preds['scores']
            sorted_idx = np.argsort(-p_scores)
            p_boxes, p_scores = p_boxes[sorted_idx], p_scores[sorted_idx]
            tp = np.zeros(len(p_boxes))
            detected_gt = np.zeros(len(targets))
            for i, p_box in enumerate(p_boxes):
                best_iou, best_gt_idx = 0, -1
                for j, t_box in enumerate(targets):
                    iou = self.compute_iou(p_box, t_box)
                    if iou > best_iou: best_iou, best_gt_idx = iou, j
                if best_iou >= iou_threshold and best_gt_idx >= 0 and not detected_gt[best_gt_idx]:
                    tp[i] = 1; detected_gt[best_gt_idx] = 1
            all_scores.extend(p_scores); all_tps.extend(tp)
        if total_gt == 0 or len(all_scores) == 0: return 0.0, 0.0, 0.0
        all_scores = np.array(all_scores); all_tps = np.array(all_tps)
        sorted_indices = np.argsort(-all_scores)
        all_tps = all_tps[sorted_indices]
        cum_tp = np.cumsum(all_tps); cum_fp = np.cumsum(1 - all_tps)
        precision = cum_tp / (cum_tp + cum_fp + 1e-16)
        recall = cum_tp / (total_gt + 1e-16)
        mrec = np.concatenate(([0.0], recall, [1.0]))
        mpre = np.concatenate(([1.0], precision, [0.0]))
        mpre = np.maximum.accumulate(mpre[::-1])[::-1]
        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
        f1 = 2 * precision * recall / (precision + recall + 1e-16)
        best_idx = np.argmax(f1) if len(f1) > 0 else 0
        return ap, precision[best_idx], recall[best_idx]
    def compute_metrics(self):
        ap50, p50, r50 = self.compute_ap(0.5)
        aps = [self.compute_ap(thr)[0] for thr in np.arange(0.5, 0.96, 0.05)]
        map50_95 = np.mean(aps)
        f1 = 2 * p50 * r50 / (p50 + r50 + 1e-16)
        return type('obj', (object,), {'box': type('obj', (object,), {'map50': ap50, 'map': map50_95, 'mp': p50, 'mr': r50, 'f1': f1})})


def load_ground_truth(label_path):
    gt_boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) >= 5:
                    xc, yc, w, h = parts[1:5] # class is 0 (first)
                    gt_boxes.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
    return np.array(gt_boxes) if gt_boxes else np.array([])


In [3]:
from ensemble_boxes import weighted_boxes_fusion
import cv2
import numpy as np

def predict_with_tta(model, img_path, conf=0.25, scales=[0.9, 1.0, 1.1], use_flip=True):
    img = cv2.imread(str(img_path))
    if img is None: return np.array([]), np.array([]), np.array([])
    orig_h, orig_w = img.shape[:2]
    all_boxes = []
    all_scores = []
    all_labels = []
    weights_tta = []
    
    variants = []
    for scale in scales:
        new_w = int(orig_w * scale)
        new_h = int(orig_h * scale)
        scaled = cv2.resize(img, (new_w, new_h))
        variants.append(('original', scale, scaled))
        if use_flip:
            flipped = cv2.flip(scaled, 1)
            variants.append(('flipped', scale, flipped))
    
    for var_type, scale, var_img in variants:
        results = model.predict(var_img, conf=conf, verbose=False)
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes.xyxyn.cpu().numpy()
            scores = results[0].boxes.conf.cpu().numpy()
            labels = results[0].boxes.cls.cpu().numpy()
            if var_type == 'flipped':
                boxes_corrected = boxes.copy()
                boxes_corrected[:, 0] = 1.0 - boxes[:, 2]
                boxes_corrected[:, 2] = 1.0 - boxes[:, 0]
                boxes = boxes_corrected
            all_boxes.append(boxes)
            all_scores.append(scores)
            all_labels.append(labels)
            weights_tta.append(1.0 if scale == 1.0 else 0.8)
        else:
            all_boxes.append(np.array([]).reshape(0, 4))
            all_scores.append(np.array([]))
            all_labels.append(np.array([]))
            weights_tta.append(0.8)
            
    if len(all_boxes) > 0 and any(len(b) > 0 for b in all_boxes):
        all_boxes = [b if len(b) > 0 else np.array([]).reshape(0, 4) for b in all_boxes]
        try:
            boxes_fused, scores_fused, labels_fused = weighted_boxes_fusion(
                all_boxes, all_scores, all_labels, weights=weights_tta, iou_thr=0.5, skip_box_thr=0.001
            )
            return boxes_fused, scores_fused, labels_fused
        except:
            pass
    return np.array([]), np.array([]), np.array([])

class MAPCalculator:
    def __init__(self):
        self.predictions = []
        self.targets = []
    def update(self, pred_boxes, pred_scores, target_boxes):
        self.predictions.append({'boxes': pred_boxes, 'scores': pred_scores})
        self.targets.append(target_boxes)
    def compute_iou(self, box1, box2):
        x1 = max(box1[0], box2[0])
        y1 = max(box1[1], box2[1])
        x2 = min(box1[2], box2[2])
        y2 = min(box1[3], box2[3])
        inter_area = max(0, x2 - x1) * max(0, y2 - y1)
        union_area = (box1[2]-box1[0])*(box1[3]-box1[1]) + (box2[2]-box2[0])*(box2[3]-box2[1]) - inter_area
        return inter_area / union_area if union_area > 0 else 0
    def compute_ap(self, iou_threshold):
        all_scores, all_tps, total_gt = [], [], 0
        for preds, targets in zip(self.predictions, self.targets):
            total_gt += len(targets)
            if len(preds['boxes']) == 0: continue
            p_boxes, p_scores = preds['boxes'], preds['scores']
            sorted_idx = np.argsort(-p_scores)
            p_boxes, p_scores = p_boxes[sorted_idx], p_scores[sorted_idx]
            tp = np.zeros(len(p_boxes))
            detected_gt = np.zeros(len(targets))
            for i, p_box in enumerate(p_boxes):
                best_iou, best_gt_idx = 0, -1
                for j, t_box in enumerate(targets):
                    iou = self.compute_iou(p_box, t_box)
                    if iou > best_iou: best_iou, best_gt_idx = iou, j
                if best_iou >= iou_threshold and best_gt_idx >= 0 and not detected_gt[best_gt_idx]:
                    tp[i] = 1; detected_gt[best_gt_idx] = 1
            all_scores.extend(p_scores); all_tps.extend(tp)
        if total_gt == 0 or len(all_scores) == 0: return 0.0, 0.0, 0.0
        all_scores = np.array(all_scores); all_tps = np.array(all_tps)
        sorted_indices = np.argsort(-all_scores)
        all_tps = all_tps[sorted_indices]
        cum_tp = np.cumsum(all_tps); cum_fp = np.cumsum(1 - all_tps)
        precision = cum_tp / (cum_tp + cum_fp + 1e-16)
        recall = cum_tp / (total_gt + 1e-16)
        mrec = np.concatenate(([0.0], recall, [1.0]))
        mpre = np.concatenate(([1.0], precision, [0.0]))
        mpre = np.maximum.accumulate(mpre[::-1])[::-1]
        i = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
        f1 = 2 * precision * recall / (precision + recall + 1e-16)
        best_idx = np.argmax(f1) if len(f1) > 0 else 0
        return ap, precision[best_idx], recall[best_idx]
    def compute_metrics(self):
        ap50, p50, r50 = self.compute_ap(0.5)
        aps = [self.compute_ap(thr)[0] for thr in np.arange(0.5, 0.96, 0.05)]
        map50_95 = np.mean(aps)
        f1 = 2 * p50 * r50 / (p50 + r50 + 1e-16)
        return type('obj', (object,), {'box': type('obj', (object,), {'map50': ap50, 'map': map50_95, 'mp': p50, 'mr': r50, 'f1': f1})})


def load_ground_truth(label_path):
    gt_boxes = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = list(map(float, line.strip().split()))
                if len(parts) >= 5:
                    xc, yc, w, h = parts[1:5] # class is 0 (first)
                    gt_boxes.append([xc-w/2, yc-h/2, xc+w/2, yc+h/2])
    return np.array(gt_boxes) if gt_boxes else np.array([])


In [4]:
import os
import json
import shutil
import requests
import zipfile
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from ultralytics import YOLO

def clear_gpu_cache():
    if torch.cuda.is_available():
        device = torch.cuda.current_device()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)
        gpu_name = torch.cuda.get_device_name(device)
        gpu_mem = torch.cuda.get_device_properties(device).total_memory / 1e9
        print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
        return True
    print("⚠️ CUDA not available")
    return False

GPU_AVAILABLE = clear_gpu_cache()
print(f"PyTorch: {torch.__version__}")

✅ GPU: NVIDIA GeForce RTX 4090 (25.4 GB)
PyTorch: 2.8.0+cu128


## 2 · Path configuration

Auto-detect the TrashCan dataset across a list of candidate locations (relative paths first, then known local absolutes) and pin the output directories. All YOLO-format artifacts are written to a dedicated `TrashCan_Filtered/` tree so the original COCO data is never modified, and the two target classes (`Metal`, `Bio`) are declared here for reporting.


In [5]:
# Smart path configuration (Local vs Server)
import os

# List of possible dataset locations (priority: relative -> local absolute)
POSSIBLE_DIRS = [
    Path("dataset"),                    # In the same folder
    Path("../dataset"),                 # In the parent folder
    Path("../../dataset"),              # Two levels up
    Path("./datasets/TrashCan/raw"),    # Default notebook structure
    Path("C:/Users/lbuln/Downloads/dataset") # Local dev
]

RAW_DIR = None
for d in POSSIBLE_DIRS:
    # Check for the 'material_version' folder, which is the key one in TrashCan
    if d.exists() and (d / "material_version").exists():
        RAW_DIR = d.resolve()
        print(f"✅ Dataset found at: {RAW_DIR}")
        break
        
if RAW_DIR is None:
    print("⚠️ No dataset found in common paths. Download will be attempted or the default path will be used.")
    RAW_DIR = Path("./datasets/TrashCan/raw").resolve()
    DOWNLOAD_NEEDED = True
else:
    DOWNLOAD_NEEDED = False
    
# YOLO output always goes to the local working directory to avoid polluting external data
# 🆕 CHANGE: Filtered directory to isolate Metal/Bio
YOLO_DIR = Path("./datasets/TrashCan_Filtered/yolo").resolve()

# Specific image and annotation directories
if (RAW_DIR / "original_data").exists():
    IMG_DIR = RAW_DIR / "original_data" 
    ANNOTATION_DIR = RAW_DIR / "material_version"
elif (RAW_DIR / "images").exists():
    IMG_DIR = RAW_DIR / "images"
    ANNOTATION_DIR = RAW_DIR / "material_version"
else:
    IMG_DIR = RAW_DIR
    ANNOTATION_DIR = RAW_DIR / "material_version"

# Create output directories
YOLO_DIR.mkdir(parents=True, exist_ok=True)
if not RAW_DIR.exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)

# Classes of interest for reporting (the actual logic lives in the conversion)
TARGET_CLASSES = ['Metal', 'Bio']

print(f"📁 Raw Data: {RAW_DIR}")
print(f"📁 Annotations: {ANNOTATION_DIR}")
print(f"📁 Images Source: {IMG_DIR}")
print(f"📁 YOLO Output: {YOLO_DIR}")


✅ Dataset found at: /home/user/work/EONSEA/Articulo-Corrosion/dataset
📁 Raw Data: /home/user/work/EONSEA/Articulo-Corrosion/dataset
📁 Annotations: /home/user/work/EONSEA/Articulo-Corrosion/dataset/material_version
📁 Images Source: /home/user/work/EONSEA/Articulo-Corrosion/dataset/original_data
📁 YOLO Output: /home/user/work/EONSEA/Articulo-Corrosion/datasets/TrashCan_Filtered/yolo


## 3 · Downloading the TrashCan dataset

The TrashCan 1.0 dataset is mirrored across several public sources. If a local copy is already present (detected in the previous step) the download is skipped; otherwise the routine attempts each mirror in order of preference.

- Official GitHub: https://github.com/pedropro/TACO
- Dataset Ninja: https://datasetninja.com/trash-can
- Zenodo mirrors


In [6]:
def download_trashcan_dataset(output_dir):
    """
    Download the TrashCan dataset.
    Tries multiple sources in order of preference.
    """
    output_dir = Path(output_dir)
    
    # Known URLs for the TrashCan Material dataset
    # Source 1: Dataset Ninja (Supervisely format convertible to COCO)
    # Source 2: GitHub mirrors
    
    sources = [
        # Option 1: Direct download from a known repository
        {
            'name': 'TrashCan-Material (GitHub)',
            'type': 'github_release',
            'url': 'https://github.com/pedropro/TrashCan/releases/download/v1.0/material_version.zip',
        },
        # Option 2: Alternative mirror
        {
            'name': 'TrashCan (Zenodo mirror)',
            'type': 'direct',
            'url': 'https://zenodo.org/record/3587843/files/trash-can.zip',
        },
    ]
    
    print("📥 Attempting to download the TrashCan dataset...")
    
    for source in sources:
        try:
            print(f"\n🔄 Trying: {source['name']}")
            
            # Download file
            zip_path = output_dir / "trashcan.zip"
            
            response = requests.get(source['url'], stream=True, timeout=30)
            if response.status_code == 200:
                total_size = int(response.headers.get('content-length', 0))
                
                with open(zip_path, 'wb') as f:
                    with tqdm(total=total_size, unit='B', unit_scale=True, desc='Downloading') as pbar:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                            pbar.update(len(chunk))
                
                # Extract
                print("📦 Extracting...")
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(output_dir)
                
                # Clean up zip
                zip_path.unlink()
                
                print(f"✅ Downloaded from: {source['name']}")
                return True
            else:
                print(f"   ⚠️ HTTP {response.status_code}")
                
        except Exception as e:
            print(f"   ❌ Error: {e}")
            continue
    
    print("\n⚠️ Automatic download failed.")
    print("   Please download manually from:")
    print("   - https://conservancy.umn.edu/handle/11299/214865")
    print(f"   And extract into: {output_dir}")
    return False

# Check whether it already exists
existing_data = list(RAW_DIR.rglob("*.json")) + list(RAW_DIR.rglob("*.jpg"))

if len(existing_data) > 100:
    print(f"✅ Dataset already present at {RAW_DIR} ({len(existing_data)} files)")
    DOWNLOAD_SUCCESS = True
elif 'DOWNLOAD_NEEDED' in locals() and DOWNLOAD_NEEDED and not list(RAW_DIR.rglob('*.json')):
    # Only download if needed and no data is present
    DOWNLOAD_SUCCESS = download_trashcan_dataset(RAW_DIR)
else:
    DOWNLOAD_SUCCESS = True
    print("✅ Dataset available (detected locally or previously downloaded).")


✅ Dataset already present at /home/user/work/EONSEA/Articulo-Corrosion/dataset (28852 files)


In [7]:
# Inspeccionar estructura descargada
def inspect_directory(path, max_depth=2):
    path = Path(path)
    print(f"\n📁 Structure of {path}:")
    
    for root, dirs, files in os.walk(path):
        depth = str(root).replace(str(path), '').count(os.sep)
        if depth < max_depth:
            indent = "  " * depth
            print(f"{indent}📂 {Path(root).name}/")
            
            # Show JSON files (annotations)
            json_files = [f for f in files if f.endswith('.json')]
            for jf in json_files[:3]:
                print(f"{indent}  📄 {jf}")
            if len(json_files) > 3:
                print(f"{indent}  ... and {len(json_files)-3} more JSON")
            
            # Count images
            img_files = [f for f in files if f.lower().endswith(('.jpg', '.png'))]
            if img_files:
                print(f"{indent}  🖼️ {len(img_files)} images")

inspect_directory(RAW_DIR)


📁 Structure of /home/user/work/EONSEA/Articulo-Corrosion/dataset:
📂 dataset/
  📂 instance_version/
    📄 instances_train_trashcan.json
    📄 instances_val_trashcan.json
  📂 original_data/
  📂 scripts/
  📂 material_version/
    📄 instances_train_trashcan.json
    📄 instances_val_trashcan.json


## 4 · COCO → YOLO conversion (critical step)

The TrashCan annotations ship in **COCO** format and must be converted to the **YOLO** text format, while filtering the original 16 categories down to the two classes of interest (`Metal`, `Bio`). This is the step that defines the cross-domain task.

**Conversion logic**
1. **COCO format:** `{"annotations": [{"bbox": [x, y, width, height], "category_id": N}]}`
2. **YOLO format:** `class_id x_center y_center width height` (normalized to 0–1)

**Normalization formulas**
```
x_center = (x + width/2) / img_width
y_center = (y + height/2) / img_height
w_norm   = width  / img_width
h_norm   = height / img_height
```


In [8]:
def find_coco_annotations(base_path):
    """
    Search the directory for COCO annotation files.
    Returns a dict of {split: annotation_path}.
    """
    base_path = Path(base_path)
    annotations = {}
    
    # Search for JSON files containing COCO annotations
    json_files = list(base_path.rglob("*.json"))
    
    for jf in json_files:
        try:
            with open(jf, 'r') as f:
                data = json.load(f)
            
            # Check whether it is COCO format (has 'images' and 'annotations')
            if 'images' in data and 'annotations' in data:
                # Determine the split based on file/directory name
                name = jf.stem.lower()
                if 'train' in name or 'train' in str(jf.parent).lower():
                    split = 'train'
                elif 'val' in name or 'valid' in name:
                    split = 'val'
                elif 'test' in name:
                    split = 'test'
                else:
                    split = 'all'
                
                annotations[split] = {
                    'path': jf,
                    'n_images': len(data['images']),
                    'n_annotations': len(data['annotations']),
                    'categories': data.get('categories', [])
                }
                print(f"   ✅ {split}: {jf.name} ({len(data['images'])} imgs, {len(data['annotations'])} anns)")
                
        except Exception as e:
            continue
    
    return annotations

print("🔍 Searching for COCO annotations...")
COCO_ANNOTATIONS = find_coco_annotations(ANNOTATION_DIR)

if not COCO_ANNOTATIONS:
    print("\n⚠️ No COCO annotations were found.")
    print("   Verify that the dataset was downloaded correctly.")

🔍 Searching for COCO annotations...
   ✅ train: instances_train_trashcan.json (6008 imgs, 9741 anns)
   ✅ val: instances_val_trashcan.json (1204 imgs, 2595 anns)


In [9]:
def convert_coco_to_yolo_filtered(coco_json_path, images_source_dir, output_dir, 
                                  target_classes, split_name='train'):
    """
    Convert COCO annotations to YOLO format with STRICT FILTERING and a custom MAPPING.
    Goal: Metal (0) vs Bio (1). Everything else is ignored.
    """
    output_dir = Path(output_dir)
    images_source_dir = Path(images_source_dir)
    
    # Create output directories
    img_out = output_dir / 'images' / split_name
    lbl_out = output_dir / 'labels' / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    
    # Load COCO annotations
    print(f"\n📄 Loading: {coco_json_path}")
    with open(coco_json_path, 'r') as f:
        coco_data = json.load(f)
    
    # Define the strict mapping
    # Original TrashCan classes:
    # rov, plant, animal_fish, animal_starfish, animal_shells, animal_crab, animal_eel, animal_etc
    # trash_etc, trash_fabric, trash_fishing_gear, trash_metal, trash_paper, trash_plastic, trash_rubber, trash_wood
    
    BIO_CLASSES = ['plant', 'animal_fish', 'animal_starfish', 'animal_shells', 'animal_crab', 'animal_eel', 'animal_etc']
    METAL_CLASSES = ['trash_metal']
    
    # Map category_id_COCO -> class_id_YOLO
    coco_cat_map = {}
    
    for cat in coco_data['categories']:
        name = cat['name']
        cid = cat['id']
        if name in METAL_CLASSES:
            coco_cat_map[cid] = 0  # Metal
        elif name in BIO_CLASSES:
            coco_cat_map[cid] = 1  # Bio
        # The rest is ignored (not added to the map)

    yolo_class_names = ['Metal', 'Bio']
    print(f"   ✅ Mapping configured: Metal (ID 0) and Bio (ID 1)")
    print(f"   ⚠️ Plastics, rubber, wood, ROV, etc. will be ignored.")
    
    # Build COCO image lookup
    images_info = {img['id']: img for img in coco_data['images']}
    
    # Group VALID annotations by image
    annotations_by_image = defaultdict(list)
    for ann in coco_data['annotations']:
        if ann['category_id'] in coco_cat_map:
            annotations_by_image[ann['image_id']].append(ann)
    
    # Statistics
    stats = {
        'total_images': 0,
        'images_with_annotations': 0,
        'background_images': 0,
        'total_annotations': 0,
        'class_counts': defaultdict(int),
        'skipped_images': 0,
    }
    
    # Process images
    print(f"\n🔄 Converting {len(images_info)} images (with strict filtering)...")
    
    for img_id, img_info in tqdm(images_info.items(), desc="Processing"):
        file_name = img_info['file_name']
        img_width = img_info['width']
        img_height = img_info['height']
        
        # Locate the physical image
        possible_paths = [
            images_source_dir / file_name,
            images_source_dir / Path(file_name).name,
            *list(images_source_dir.rglob(Path(file_name).name)),
        ]
        
        img_path = None
        for pp in possible_paths:
            if pp.exists():
                img_path = pp
                break
        
        if not img_path:
            stats['skipped_images'] += 1
            continue
        
        # Copy image
        dest_img = img_out / img_path.name
        if not dest_img.exists():
            shutil.copy(img_path, dest_img)
        
        stats['total_images'] += 1
        
        # Convert labels
        img_annotations = annotations_by_image.get(img_id, [])
        label_file = lbl_out / (img_path.stem + '.txt')
        
        if img_annotations:
            stats['images_with_annotations'] += 1
            with open(label_file, 'w') as f:
                for ann in img_annotations:
                    bbox = ann['bbox']
                    # YOLO normalization
                    x_center = (bbox[0] + bbox[2] / 2) / img_width
                    y_center = (bbox[1] + bbox[3] / 2) / img_height
                    w = bbox[2] / img_width
                    h = bbox[3] / img_height
                    
                    # Mapped ID
                    yolo_class = coco_cat_map[ann['category_id']]
                    class_name = yolo_class_names[yolo_class]
                    
                    f.write(f"{yolo_class} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")
                    
                    stats['total_annotations'] += 1
                    stats['class_counts'][class_name] += 1
        else:
            # Image with neither Metal nor Bio -> Background (negative)
            stats['background_images'] += 1
            label_file.touch()
    
    print(f"\n📊 Statistics {split_name}:")
    print(f"   Images processed: {stats['total_images']}")
    print(f"   With annotations: {stats['images_with_annotations']}")
    print(f"   Background (no target): {stats['background_images']}")
    print(f"   Per class: {dict(stats['class_counts'])}")
    
    return stats, yolo_class_names


In [10]:
# Run conversion for each split found
all_stats = {}
CLASS_NAMES = None

if COCO_ANNOTATIONS:
    for split, info in COCO_ANNOTATIONS.items():
        stats, class_names = convert_coco_to_yolo_filtered(
            coco_json_path=info['path'],
            images_source_dir=IMG_DIR,
            output_dir=YOLO_DIR,
            target_classes=TARGET_CLASSES,
            split_name=split if split != 'all' else 'train'
        )
        all_stats[split] = stats
        if CLASS_NAMES is None:
            CLASS_NAMES = class_names
else:
    print("❌ No annotations to convert.")
    print("   Using fallback dataset...")
    
    # Fallback: use the existing local dataset
    local_dataset = Path("./dataset_yolo")
    if local_dataset.exists():
        YOLO_DIR = local_dataset
        CLASS_NAMES = ['corrosion']
        print(f"   📁 Using: {YOLO_DIR}")


📄 Loading: /home/user/work/EONSEA/Articulo-Corrosion/dataset/material_version/instances_train_trashcan.json
   ✅ Mapping configured: Metal (ID 0) and Bio (ID 1)
   ⚠️ Plastics, rubber, wood, ROV, etc. will be ignored.

🔄 Converting 6008 images (with strict filtering)...


Processing: 100%|██████████| 6008/6008 [00:36<00:00, 164.18it/s]



📊 Statistics train:
   Images processed: 6008
   With annotations: 2264
   Background (no target): 3744
   Per class: {'Bio': 2155, 'Metal': 901}

📄 Loading: /home/user/work/EONSEA/Articulo-Corrosion/dataset/material_version/instances_val_trashcan.json
   ✅ Mapping configured: Metal (ID 0) and Bio (ID 1)
   ⚠️ Plastics, rubber, wood, ROV, etc. will be ignored.

🔄 Converting 1204 images (with strict filtering)...


Processing: 100%|██████████| 1204/1204 [00:07<00:00, 170.69it/s]


📊 Statistics val:
   Images processed: 1204
   With annotations: 525
   Background (no target): 679
   Per class: {'Bio': 650, 'Metal': 225}


In [11]:
# Create data.yaml for YOLO
def create_yolo_yaml(output_dir, class_names):
    output_dir = Path(output_dir)
    yaml_path = output_dir / 'data.yaml'
    
    # Check available splits
    train_path = output_dir / 'images' / 'train'
    val_path = output_dir / 'images' / 'val'
    test_path = output_dir / 'images' / 'test'
    
    # If no val split exists, create one from train (80/20)
    if train_path.exists() and not val_path.exists():
        print("📊 Creating train/val split (80/20)...")
        
        import random
        random.seed(42)
        
        train_images = list(train_path.glob('*'))
        random.shuffle(train_images)
        
        split_idx = int(len(train_images) * 0.8)
        val_images = train_images[split_idx:]
        
        # Move to val
        val_path.mkdir(parents=True, exist_ok=True)
        val_labels = output_dir / 'labels' / 'val'
        val_labels.mkdir(parents=True, exist_ok=True)
        
        for img in val_images:
            shutil.move(str(img), val_path / img.name)
            lbl = output_dir / 'labels' / 'train' / (img.stem + '.txt')
            if lbl.exists():
                shutil.move(str(lbl), val_labels / lbl.name)
        
        print(f"   Train: {split_idx}, Val: {len(val_images)}")
    
    # Generate YAML
    yaml_content = f"""# TrashCan Material Dataset - Auto-generated
# Filtered for classes: {class_names}

path: {output_dir.absolute()}
train: images/train
val: images/val

nc: {len(class_names)}

names:
"""
    for i, name in enumerate(class_names):
        yaml_content += f"  {i}: {name}\n"
    
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    
    print(f"\n✅ YAML created: {yaml_path}")
    print(yaml_content)
    
    return yaml_path

if CLASS_NAMES:
    YAML_PATH = create_yolo_yaml(YOLO_DIR, CLASS_NAMES)
else:
    # Search for an existing yaml
    yaml_files = list(YOLO_DIR.rglob('*.yaml'))
    if yaml_files:
        YAML_PATH = yaml_files[0]
        print(f"📄 Using existing YAML: {YAML_PATH}")
    else:
        raise FileNotFoundError("data.yaml not found")


✅ YAML created: /home/user/work/EONSEA/Articulo-Corrosion/datasets/TrashCan_Filtered/yolo/data.yaml
# TrashCan Material Dataset - Auto-generated
# Filtered for classes: ['Metal', 'Bio']

path: /home/user/work/EONSEA/Articulo-Corrosion/datasets/TrashCan_Filtered/yolo
train: images/train
val: images/val

nc: 2

names:
  0: Metal
  1: Bio



## 5 · Model configuration

Training hyper-parameters for the Medium backbone (`yolo26m.pt`): 50 epochs, batch size 32, 640 px input, early-stopping patience 15. These mirror the Direct Transfer recipe used for the underwater corrosion model so the comparison across domains is controlled.


In [12]:
CONFIG = {
    'model': 'yolo26m.pt',  # Medium
    'epochs': 50,
    'batch': 32,
    'imgsz': 640,
    'patience': 15,
    'lr0': 0.001,
    'lrf': 0.01,
    'project': 'TrashCan_Filtered_Validation',  # New project
    'name': 'yolo_medium_metal_bio_strict',     # New name
}

print("📋 Configuration:")
for k, v in CONFIG.items():
    print(f"   {k}: {v}")

📋 Configuration:
   model: yolo26m.pt
   epochs: 50
   batch: 32
   imgsz: 640
   patience: 15
   lr0: 0.001
   lrf: 0.01
   project: TrashCan_Filtered_Validation
   name: yolo_medium_metal_bio_strict


## 6 · Training

Retrain `yolo26m.pt` on the filtered TrashCan dataset. The full Ultralytics training trace below (50 epochs) is preserved as executed; the final fused-model validation reports per-class Precision/Recall/mAP for `Metal` and `Bio`.


In [13]:
clear_gpu_cache()

print(f"🚀 Loading: {CONFIG['model']}")
model = YOLO(CONFIG['model'])

print("🏋️ Training on TrashCan...")
results = model.train(
    data=str(YAML_PATH),
    epochs=CONFIG['epochs'],
    batch=CONFIG['batch'],
    imgsz=CONFIG['imgsz'],
    patience=CONFIG['patience'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    project=CONFIG['project'],
    name=CONFIG['name'],
    exist_ok=True,
    verbose=True,
    plots=True,
)

print("✅ Training complete!")

✅ GPU: NVIDIA GeForce RTX 4090 (25.4 GB)
🚀 Loading: yolo26m.pt
🏋️ Training on TrashCan...
New https://pypi.org/project/ultralytics/8.4.7 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/user/work/EONSEA/Articulo-Corrosion/datasets/TrashCan_Filtered/yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, 

In [14]:
BEST_WEIGHTS = Path(CONFIG['project']) / CONFIG['name'] / 'weights' / 'best.pt'
trained_model = YOLO(str(BEST_WEIGHTS))
print(f"📦 Model loaded: {BEST_WEIGHTS}")

📦 Model loaded: TrashCan_Filtered_Validation/yolo_medium_metal_bio_strict/weights/best.pt


## 7 · TTA experiment: Standard vs Augmented

The core experiment. The trained model is evaluated twice on the validation split: once with **standard** single-pass inference (`augment=False`), and once with **manual TTA** — predictions over three scales (0.9, 1.0, 1.1) and a horizontal flip, fused with Weighted Boxes Fusion. The custom `MAPCalculator` reproduces the COCO-style mAP/Precision/Recall/F1 so both settings are scored on an identical footing.


In [15]:
# STANDARD VALIDATION
print("="*60)
print("📊 STANDARD VALIDATION (augment=False)")
print("="*60)

metrics_std = trained_model.val(
    data=str(YAML_PATH),
    split='val',
    augment=False,
    name='val_standard',
    project=CONFIG['project'],
    exist_ok=True,
)

# Per-class metrics
print(f"\n📈 Standard Results (average):")
print(f"   mAP@50:    {metrics_std.box.map50:.4f}")
print(f"   mAP@50-95: {metrics_std.box.map:.4f}")
print(f"   Precision: {metrics_std.box.mp:.4f}")
print(f"   Recall:    {metrics_std.box.mr:.4f}")

# Per-class F1
f1_std = 2 * metrics_std.box.mp * metrics_std.box.mr / (metrics_std.box.mp + metrics_std.box.mr + 1e-6)
print(f"   F1-Score:  {f1_std:.4f}")

📊 STANDARD VALIDATION (augment=False)
Ultralytics 8.4.5 🚀 Python-3.9.13 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24215MiB)
YOLO26m summary (fused): 132 layers, 20,350,994 parameters, 0 gradients, 67.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1640.6±621.9 MB/s, size: 22.5 KB)
val: Scanning /home/user/work/EONSEA/Articulo-Corrosion/datasets/TrashCan_Filtered/yolo/labels/val.cache... 1204 images, 679 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1204/1204 229.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 76/76 22.4it/s 3.4s0.1s
                   all       1204        875      0.732       0.63      0.664      0.453
                 Metal        214        225      0.833      0.643      0.699      0.525
                   Bio        369        650       0.63      0.617      0.629      0.381
Speed: 0.3ms preprocess, 1.7ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /home/u

In [16]:
# MANUAL TTA VALIDATION
print("="*60)
print("📊 TTA VALIDATION (Manual: Scales + Flip)")
print("="*60)

calc_tta = MAPCalculator()
val_images = list((YOLO_DIR / 'images' / 'val').glob('*'))
val_labels_dir = YOLO_DIR / 'labels' / 'val'

print(f"Evaluating {len(val_images)} images with TTA...")

for img_path in tqdm(val_images, desc="TTA Eval"):
    # 1. TTA prediction
    boxes, scores, labels = predict_with_tta(
        trained_model, img_path, conf=0.001, # Very low conf for mAP calculation
        scales=[0.9, 1.0, 1.1], use_flip=True
    )
    
    # 2. Load GT
    label_path = val_labels_dir / (img_path.stem + '.txt')
    gt_boxes = load_ground_truth(label_path)
    
    # 3. Update metrics
    if len(boxes) > 0:
        calc_tta.update(boxes, scores, gt_boxes)
    elif len(gt_boxes) > 0:
        calc_tta.update(np.array([]).reshape(0,4), np.array([]), gt_boxes)

metrics_tta = calc_tta.compute_metrics()

print(f"\n📈 TTA Results (average):")
print(f"   mAP@50:    {metrics_tta.box.map50:.4f}")
print(f"   mAP@50-95: {metrics_tta.box.map:.4f}")
print(f"   Precision: {metrics_tta.box.mp:.4f}")
print(f"   Recall:    {metrics_tta.box.mr:.4f}")

f1_tta = metrics_tta.box.f1 # Already calculated in wrapper
print(f"   F1-Score:  {f1_tta:.4f}")


📊 TTA VALIDATION (Manual: Scales + Flip)
Evaluating 1204 images with TTA...


TTA Eval: 100%|██████████| 1204/1204 [00:54<00:00, 22.01it/s]



📈 TTA Results (average):
   mAP@50:    0.6461
   mAP@50-95: 0.3906
   Precision: 0.7077
   Recall:    0.6697
   F1-Score:  0.6882


In [17]:
# Comparison table
results_df = pd.DataFrame({
    'Metric': ['mAP@50', 'mAP@50-95', 'Precision', 'Recall', 'F1-Score'],
    'Standard': [metrics_std.box.map50, metrics_std.box.map, 
                 metrics_std.box.mp, metrics_std.box.mr, f1_std],
    'TTA': [metrics_tta.box.map50, metrics_tta.box.map,
            metrics_tta.box.mp, metrics_tta.box.mr, f1_tta],
})
results_df['Δ'] = results_df['TTA'] - results_df['Standard']
results_df['Improvement %'] = (results_df['Δ'] / results_df['Standard'] * 100).round(2)

print("\n" + "="*70)
print("📊 FINAL COMPARISON: STANDARD vs TTA")
print("="*70)
print(results_df.to_string(index=False, float_format='{:.4f}'.format))

results_df.to_csv('trashcan_tta_results.csv', index=False)
print("\n💾 Saved: trashcan_tta_results.csv")


📊 FINAL COMPARISON: STANDARD vs TTA
    Metric  Standard    TTA       Δ  Improvement %
   mAP@50    0.6636 0.6461 -0.0175   -2.6400
mAP@50-95    0.4531 0.3906 -0.0625  -13.7900
Precision    0.7316 0.7077 -0.0239   -3.2700
   Recall    0.6302 0.6697  0.0395    6.2800
 F1-Score    0.6771 0.6882  0.0111    1.6400

💾 Saved: trashcan_tta_results.csv


## 8 · Prediction visualization

Qualitative check: TTA-fused detections (multi-scale + flip) overlaid on a sample of eight validation images, plus the Standard-vs-TTA bar chart of the five summary metrics.


In [18]:
# Show example predictions WITH MANUAL TTA
val_images = list((YOLO_DIR / 'images' / 'val').glob('*'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, img_path in enumerate(val_images):
    if idx >= 8: break
    
    # Predict with TTA
    boxes, scores, labels = predict_with_tta(
        trained_model, img_path, 
        conf=0.25, # Visual confidence
        scales=[0.9, 1.0, 1.1], use_flip=True
    )
    
    # Plotting
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box # normalized
        h, w = img.shape[:2]
        x1, x2 = x1*w, x2*w
        y1, y2 = y1*h, y2*h
        
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img, f"{score:.2f}", (int(x1), int(y1)-5), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    axes[idx].imshow(img)
    axes[idx].set_title(img_path.stem[:20], fontsize=9)
    axes[idx].axis('off')

plt.suptitle('TrashCan Predictions (Manual TTA Enabled)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('trashcan_predictions_tta.png', dpi=150)
plt.show()


<Figure size 1600x800 with 8 Axes>

In [19]:
# Comparison plot
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['Standard'], width, label='Standard', color='#3498db')
ax.bar(x + width/2, results_df['TTA'], width, label='TTA', color='#e74c3c')

ax.set_xticks(x)
ax.set_xticklabels(results_df['Metric'])
ax.set_ylim(0, 1.1)
ax.legend()
ax.set_title('TrashCan Validation: Standard vs TTA')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('trashcan_comparison.png', dpi=150)
plt.show()

<Figure size 1000x500 with 1 Axes>

## 9 · Conclusions

Summary of the cross-domain experiment feeding the paper's `tab:trashcan_results`.

- **TTA improves Recall by +6.28%** (0.6302 → 0.6697) and lifts **F1-Score from 0.6771 to 0.6882**, at the cost of a small drop in Precision (−3.27%) and mAP@50 (−2.64%).
- Because the *Metal* class is a proxy for submerged corrosion, this result shows the augmentation strategy **transfers across underwater domains** rather than being tuned to the SINTEF LIACI corrosion set.
- The recall-oriented gain is the operationally relevant one for inspection tasks, where missing a corroded/metallic instance is costlier than a marginal precision loss.


In [20]:
print("="*70)
print("📋 SUMMARY: TRASHCAN VALIDATION (METAL DETECTION)")
print("="*70)

print(f"\n📊 Dataset: TrashCan Material")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Model: {CONFIG['model']}")

f1_change = (f1_tta - f1_std) / f1_std * 100 if f1_std > 0 else 0
map_change = (metrics_tta.box.map50 - metrics_std.box.map50) / metrics_std.box.map50 * 100 if metrics_std.box.map50 > 0 else 0

print(f"\n📈 RESULTS:")
print(f"   F1 Standard: {f1_std:.4f}")
print(f"   F1 TTA:      {f1_tta:.4f} ({'+' if f1_change > 0 else ''}{f1_change:.1f}%)")
print(f"   mAP Standard: {metrics_std.box.map50:.4f}")
print(f"   mAP TTA:      {metrics_tta.box.map50:.4f} ({'+' if map_change > 0 else ''}{map_change:.1f}%)")

if f1_change > 0:
    print(f"\n✅ TTA IMPROVES performance on TrashCan")
else:
    print(f"\n⚠️ TTA does not significantly improve on this dataset")

print("\n" + "="*70)

📋 SUMMARY: TRASHCAN VALIDATION (METAL DETECTION)

📊 Dataset: TrashCan Material
   Classes: ['Metal', 'Bio']
   Model: yolo26m.pt

📈 RESULTS:
   F1 Standard: 0.6771
   F1 TTA:      0.6882 (+1.6%)
   mAP Standard: 0.6636
   mAP TTA:      0.6461 (-2.6%)

✅ TTA IMPROVES performance on TrashCan

